In [0]:
%sql
I) Cloud Object storage (S3, ADLS, GCS, Volume)
Volume:
csv,json,parquet, avro, txt, pdf, excel, xml, xls...etc

a. Batch 
          1. sql ==== CTAS read_files()
          2. UI ( volume )
          3.pyspark=== df 

In [0]:
%sql
read_files is a TVF from databricks. In built function

In [0]:
%sql
select * from read_files("/Volumes/dev/naval/raw/financial_dataset.csv")

In [0]:
%sql
create table dev.naval.financail_sql as 
select * from read_files("/Volumes/dev/naval/raw/financial_dataset.csv")

In [0]:
%sql
select * from dev.naval.financail_sql

In [0]:
%sql
desc extended dev.naval.financail_sql

In [0]:
%sql
desc detail dev.naval.financail_sql

In [0]:
%sql
Task : create table using json : 



/Volumes/dev/naval/raw/drivers.json

In [0]:
%sql
create or replace table dev.naval.drivers_sql as 
select *,current_timestamp()  as ingestion_date, _metadata.file_name as path from read_files("/Volumes/dev/naval/raw/drivers.json")

In [0]:
%sql
create table if not exists dev.naval.drivers_sql as 
select *,current_timestamp()  as ingestion_date, _metadata.file_name as path from read_files("/Volumes/dev/naval/raw/drivers.json")

In [0]:
%sql
select *,current_timestamp()  as ingestion_date, _metadata.file_name as path from read_files("/Volumes/dev/naval/raw/drivers.json")

In [0]:
%sql
Spark

basic spark architecture
all purpose compute /serverless

create table using pyspark df

In [0]:
Lazy Evalution: 

Transformation
read

Action 
display
write

In [0]:
%python
print("hello")

In [0]:
%sql
rdd, Data frame 

In [0]:
%sql
Reader API -- Extract-- Reading

write api-- Loading -- writeing

In [0]:
%python
df=spark.read.csv("/Volumes/dev/naval/raw/financial_dataset.csv",header=True,inferSchema=True)

In [0]:
df.filter("TransactionType='Deposit'")#.display()

In [0]:
df.filter("TransactionType='Deposit'").explain()

In [0]:
%python
df.display()

In [0]:
from pyspark.sql.functions import *

In [0]:
df=spark.read.csv("/Volumes/dev/naval/raw/financial_dataset.csv",header=True,inferSchema=True)

In [0]:
df1=df.withColumn("ingestion_date",current_timestamp()).withColumn("path",col("_metadata.file_name"))

In [0]:
df1.write.mode("overwrite").saveAsTable("dev.naval.financial_pyspark")

In [0]:
df=spark.read.csv("/Volumes/dev/naval/raw/financial_dataset.csv",header=True,inferSchema=True)
df1=df.withColumn("ingestion_date",current_timestamp()).withColumn("path",col("_metadata.file_name"))
df1.write.mode("overwrite").saveAsTable("dev.naval.financial_pyspark")

In [0]:
(# reading
spark
 .read
 .csv("/Volumes/dev/naval/raw/financial_dataset.csv",header=True,inferSchema=True)

 # tranformation
 .withColumn("ingestion_date",current_timestamp())
 .withColumn("path",col("_metadata.file_name"))

 # writing / Loading
 .write
 .mode("overwrite")
 .saveAsTable("dev.naval.financial_pyspark")
 )

In [0]:
input_path="/Volumes/dev/naval/raw/financial_dataset.csv"
catalog="dev"
schema="naval"
table_name="financial_pyspark"

In [0]:
def add_columns(df):
    return df.withColumn("ingestion_date",current_timestamp()).withColumn("path",col("_metadata.file_name"))

In [0]:
df=(spark.read.csv(input_path,header=True,inferSchema=True))
df1=add_columns(df)
df1.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table_name}")

In [0]:
df=spark.table("dev.naval.financial_pyspark")

In [0]:
df.display()

In [0]:
%sql
select * from dev.naval.financial_pyspark

In [0]:
df=(spark.read.csv(input_path,header=True,inferSchema=True))

In [0]:
#df.select("*")
df.select("TransactionID","AmountUSD").display()

In [0]:
df.filter("AmountUSD>1000").display()

In [0]:
df.orderBy(col("TransactionType").desc()).display()

In [0]:
df.withColumnRenamed("TransactionID","transaction_id").withColumnRenamed("CustomerID","customer_id")

In [0]:
df.withColumnsRenamed({"TransactionID":"transaction_id","CustomerID":"customer_id"})

In [0]:
df.printSchema()

In [0]:
df_lower = df.toDF(*[col_name.lower() for col_name in df.columns])
df_lower.display()

In [0]:
df_lower.groupBy("branch").count().display()

In [0]:
df_lower.groupBy("branch").count().sum("amountusd").display()

In [0]:
df_lower.withColumn("env",lit("dev")).display()

In [0]:
df_lower.groupBy("branch").agg(count("*"), 
                                       sum("amountusd").alias("sum"),
                                       min("amountusd").alias("min")).display()